# MERA入门教程

本教程介绍多尺度纠缠重整化拟设(MERA)的基本概念和实现。

## 学习目标

1. 理解MERA的层次结构
2. 掌握等距张量和解纠缠器
3. 实现简单的MERA收缩
4. 理解实空间重整化群

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrowPatch
from matplotlib.lines import Line2D
import sys

sys.path.append('../../common')
from utils.tensor_utils import entanglement_entropy

plt.style.use('seaborn-v0_8-darkgrid')
np.set_printoptions(precision=4, suppress=True)

## 1. MERA结构

### 基本构件

MERA由两种张量构成：

1. **等距张量** (Isometry): $u_{αβ}^i$, 形状 $(χ, χ, χ)$
   - 粗粒化：2个格点 → 1个格点
   - 满足 $u^\dagger u = I$

2. **解纠缠器** (Disentangler): $w_{αβ}^{ij}$, 形状 $(χ, χ, χ, χ)$
   - 去除短程纠缠
   - 满足 $w^\dagger w = I$（酉性）

In [ ]:
def visualize_mera_structure(num_layers=3):
    """可视化MERA层次结构"""
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # 参数
    layer_height = 2.5
    site_spacing = 1.0
    
    # 颜色
    isometry_color = 'lightblue'
    disentangler_color = 'lightgreen'
    
    # 绘制每一层
    for layer in range(num_layers + 1):
        y = layer * layer_height
        n_sites = 2 ** (num_layers - layer)
        
        # 物理格点
        for i in range(n_sites):
            x = (i - n_sites / 2 + 0.5) * site_spacing * (2 ** layer)
            circle = Circle((x, y), 0.15, color='orange', 
                          ec='black', linewidth=2, zorder=5)
            ax.add_patch(circle)
            
            if layer == 0:
                ax.text(x, y-0.5, f's{i}', ha='center', fontsize=9)
        
        # 解纠缠器和等距张量（如果不是最顶层）
        if layer < num_layers:
            # 解纠缠器
            y_dis = y + 0.5
            for i in range(0, n_sites, 2):
                x = (i - n_sites / 2 + 1) * site_spacing * (2 ** layer)
                rect = FancyBboxPatch((x-0.3, y_dis-0.2), 0.6, 0.4,
                                     boxstyle="round,pad=0.05",
                                     facecolor=disentangler_color,
                                     edgecolor='black', linewidth=1.5, zorder=4)
                ax.add_patch(rect)
                ax.text(x, y_dis, 'w', ha='center', va='center', 
                       fontsize=10, fontweight='bold')
                
                # 连接线
                x1 = (i - n_sites / 2 + 0.5) * site_spacing * (2 ** layer)
                x2 = (i + 1 - n_sites / 2 + 0.5) * site_spacing * (2 ** layer)
                ax.plot([x1, x1, x], [y+0.15, y_dis-0.2, y_dis-0.2], 
                       'k-', linewidth=1.5, zorder=3)
                ax.plot([x2, x2, x], [y+0.15, y_dis-0.2, y_dis-0.2],
                       'k-', linewidth=1.5, zorder=3)
            
            # 等距张量
            y_iso = y + 1.5
            for i in range(0, n_sites, 2):
                x = (i - n_sites / 2 + 1) * site_spacing * (2 ** layer)
                # 三角形（等距张量）
                triangle = plt.Polygon([(x-0.25, y_iso-0.3), 
                                       (x+0.25, y_iso-0.3),
                                       (x, y_iso+0.2)],
                                      facecolor=isometry_color,
                                      edgecolor='black', linewidth=1.5, zorder=4)
                ax.add_patch(triangle)
                ax.text(x, y_iso-0.1, 'u', ha='center', va='center',
                       fontsize=10, fontweight='bold')
                
                # 连接线
                ax.plot([x, x], [y_dis+0.2, y_iso-0.3], 
                       'k-', linewidth=1.5, zorder=3)
                ax.plot([x, x], [y_iso+0.2, y+layer_height-0.15],
                       'k-', linewidth=1.5, zorder=3)
    
    # 标注层数
    for layer in range(num_layers + 1):
        y = layer * layer_height
        ax.text(-5, y, f'Layer {layer}', fontsize=11, fontweight='bold',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 图例
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='orange',
              markersize=12, label='Site/State'),
        Line2D([0], [0], marker='^', color='w', markerfacecolor=isometry_color,
              markersize=12, label='Isometry (u)'),
        Line2D([0], [0], marker='s', color='w', markerfacecolor=disentangler_color,
              markersize=12, label='Disentangler (w)')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=11)
    
    ax.set_xlim(-6, 6)
    ax.set_ylim(-1, num_layers * layer_height + 1)
    ax.set_aspect('equal')
    ax.set_title(f'MERA Structure ({num_layers} layers, branching=2)',
                fontsize=14, fontweight='bold', pad=20)
    ax.axis('off')
    
    plt.tight_layout()
    return fig

visualize_mera_structure(num_layers=3)
plt.show()

## 2. 等距张量

### 定义

等距张量 $u_{αβ}^i$ 满足：

$$
\sum_{α,β} (u_{αβ}^i)^* u_{αβ}^j = δ_{ij}
$$

即 $u^\dagger u = I$

### 构造方法

In [ ]:
def random_isometry(chi_in, chi_out, random_state=42):
    """
    生成随机等距张量
    
    参数:
        chi_in: 输入键维度
        chi_out: 输出键维度
    
    返回:
        u: 等距张量 (chi_in, chi_in, chi_out)
    """
    np.random.seed(random_state)
    
    # 随机矩阵
    A = np.random.randn(chi_in * chi_in, chi_out) + \
        1j * np.random.randn(chi_in * chi_in, chi_out)
    
    # QR分解得到等距矩阵
    Q, R = np.linalg.qr(A)
    
    # Reshape为张量
    u = Q[:, :chi_out].reshape(chi_in, chi_in, chi_out)
    
    return u

def verify_isometry(u, verbose=True):
    """
    验证等距性
    
    检查 u† u = I
    """
    chi_in, _, chi_out = u.shape
    
    # u†u
    u_mat = u.reshape(chi_in * chi_in, chi_out)
    result = u_mat.conj().T @ u_mat
    
    # 应该是单位矩阵
    identity = np.eye(chi_out)
    error = np.linalg.norm(result - identity)
    
    if verbose:
        print(f"等距性误差: {error:.2e}")
        if error < 1e-10:
            print("✓ 等距性验证通过!")
        else:
            print("✗ 等距性验证失败!")
    
    return error < 1e-10

# 测试
print("等距张量测试")
print("="*50)

chi = 4
u = random_isometry(chi, chi)

print(f"\n等距张量形状: {u.shape}")
print(f"张量范数: {np.linalg.norm(u):.4f}")

print("\n验证等距性:")
verify_isometry(u)

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# u†u的矩阵元
u_mat = u.reshape(chi * chi, chi)
result = u_mat.conj().T @ u_mat

im1 = ax1.imshow(np.abs(result), cmap='Blues', vmin=0, vmax=1)
ax1.set_title('|u† u|', fontsize=13, fontweight='bold')
ax1.set_xlabel('Column index')
ax1.set_ylabel('Row index')
plt.colorbar(im1, ax=ax1)

# 对角元
diagonal = np.diag(result)
ax2.plot(np.abs(diagonal), 'o-', markersize=8, linewidth=2, label='|diagonal|')
ax2.axhline(1, color='red', linestyle='--', linewidth=2, label='Expected: 1')
ax2.set_xlabel('Index', fontsize=12)
ax2.set_ylabel('Value', fontsize=12)
ax2.set_title('Diagonal of u† u', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 解纠缠器

### 酉张量

解纠缠器 $w_{αβ}^{ij}$ 是酉张量：

$$
\sum_{α,β} (w_{αβ}^{ij})^* w_{αβ}^{kl} = δ_{ik}δ_{jl}
$$

即 $w^\dagger w = I$ 且 $w w^\dagger = I$

In [ ]:
def random_unitary(chi, random_state=42):
    """
    生成随机酉张量（解纠缠器）
    
    参数:
        chi: 键维度
    
    返回:
        w: 酉张量 (chi, chi, chi, chi)
    """
    np.random.seed(random_state)
    
    # 随机复矩阵
    A = np.random.randn(chi**2, chi**2) + \
        1j * np.random.randn(chi**2, chi**2)
    
    # SVD得到酉矩阵
    U, S, Vh = np.linalg.svd(A)
    W_mat = U @ Vh  # 酉矩阵
    
    # Reshape
    w = W_mat.reshape(chi, chi, chi, chi)
    
    return w

def verify_unitary(w, verbose=True):
    """
    验证酉性
    
    检查 w†w = I 和 ww† = I
    """
    chi = w.shape[0]
    w_mat = w.reshape(chi**2, chi**2)
    
    # w†w
    result1 = w_mat.conj().T @ w_mat
    error1 = np.linalg.norm(result1 - np.eye(chi**2))
    
    # ww†
    result2 = w_mat @ w_mat.conj().T
    error2 = np.linalg.norm(result2 - np.eye(chi**2))
    
    if verbose:
        print(f"酉性误差 (w†w): {error1:.2e}")
        print(f"酉性误差 (ww†): {error2:.2e}")
        if error1 < 1e-10 and error2 < 1e-10:
            print("✓ 酉性验证通过!")
        else:
            print("✗ 酉性验证失败!")
    
    return error1 < 1e-10 and error2 < 1e-10

# 测试
print("\n解纠缠器测试")
print("="*50)

chi = 3
w = random_unitary(chi)

print(f"\n解纠缠器形状: {w.shape}")
print(f"张量范数: {np.linalg.norm(w):.4f}")

print("\n验证酉性:")
verify_unitary(w)

## 4. MERA粗粒化

### RG变换

一层MERA变换：

$$
|ψ'⟩ = \prod_i u_i^\dagger \prod_j w_j^\dagger |ψ⟩
$$

系统尺寸：L → L/2

In [ ]:
def mera_coarse_graining_step(state, disentanglers, isometries):
    """
    执行一步MERA粗粒化
    
    参数:
        state: 当前层态向量
        disentanglers: 解纠缠器列表
        isometries: 等距张量列表
    
    返回:
        粗粒化后的态
    """
    # 简化实现：假设态是向量形式
    # 完整实现需要张量收缩
    
    L = len(state)
    L_new = L // 2
    
    # 应用解纠缠器（简化）
    state_dis = state.copy()
    
    # 应用等距张量（粗粒化）
    state_new = np.zeros(L_new, dtype=complex)
    
    for i in range(L_new):
        # 简化：平均相邻两个格点
        state_new[i] = (state_dis[2*i] + state_dis[2*i+1]) / np.sqrt(2)
    
    return state_new

# 演示粗粒化
print("\nMERA粗粒化演示")
print("="*50)

# 初始态
L0 = 16
state = np.random.randn(L0) + 1j * np.random.randn(L0)
state = state / np.linalg.norm(state)

print(f"\n初始系统大小: L = {L0}")
print(f"初始态范数: {np.linalg.norm(state):.4f}")

# 多层粗粒化
states = [state]
num_layers = int(np.log2(L0))

for layer in range(num_layers):
    L_current = len(states[-1])
    
    # 生成张量
    chi = 2  # 简化
    n_dis = L_current // 2
    n_iso = L_current // 2
    
    disentanglers = [random_unitary(chi, random_state=layer*100+i) 
                     for i in range(n_dis)]
    isometries = [random_isometry(chi, chi, random_state=layer*100+i)
                 for i in range(n_iso)]
    
    # 粗粒化
    state_new = mera_coarse_graining_step(states[-1], disentanglers, isometries)
    states.append(state_new)
    
    print(f"Layer {layer+1}: L = {len(state_new)}, norm = {np.linalg.norm(state_new):.4f}")

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 系统尺寸缩减
L_values = [len(s) for s in states]
ax1.plot(range(len(L_values)), L_values, 'o-', markersize=10, linewidth=2)
ax1.set_xlabel('Layer', fontsize=12)
ax1.set_ylabel('System Size L', fontsize=12)
ax1.set_title('Coarse-Graining: System Size Reduction',
             fontsize=13, fontweight='bold')
ax1.set_yscale('log', base=2)
ax1.grid(True, alpha=0.3)

# 态的幅度分布（不同层）
for i, s in enumerate(states[::2]):  # 每隔一层
    ax2.plot(np.abs(s)**2, 'o-', markersize=4, linewidth=1.5, 
            alpha=0.7, label=f'Layer {i*2}')

ax2.set_xlabel('Site Index', fontsize=12)
ax2.set_ylabel('Probability |ψ|²', fontsize=12)
ax2.set_title('State Distribution at Different Layers',
             fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 纠缠熵演化

### MERA的关键性质

对于临界系统：
- **MPS**: S(L) ~ c/3 log L (需要χ指数增长)
- **MERA**: S(L) ~ log log L (χ多项式即可)

MERA高效！

In [ ]:
def compare_entanglement_scaling():
    """比较MPS和MERA的纠缠熵标度"""
    L_values = 2 ** np.arange(2, 11)  # 4, 8, 16, ..., 1024
    
    # 中心荷
    c = 0.5  # Ising CFT
    
    # MPS纠缠熵（临界点）
    S_mps = (c / 3) * np.log(L_values) + 0.1 * np.random.randn(len(L_values))
    
    # MERA纠缠熵
    S_mera = np.log(np.log(L_values)) + 0.5 + 0.05 * np.random.randn(len(L_values))
    
    # 可视化
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 纠缠熵 vs L
    ax1.plot(L_values, S_mps, 'o-', markersize=8, linewidth=2, label='MPS')
    ax1.plot(L_values, S_mera, 's-', markersize=8, linewidth=2, label='MERA')
    ax1.set_xlabel('System Size L', fontsize=12)
    ax1.set_ylabel('Entanglement Entropy S', fontsize=12)
    ax1.set_title('Entanglement Scaling at Criticality',
                 fontsize=13, fontweight='bold')
    ax1.set_xscale('log')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # 所需键维度（估计）
    # MPS: χ ~ exp(S)
    # MERA: χ ~ poly(log L)
    chi_mps = np.exp(S_mps)
    chi_mera = (np.log(L_values))**2  # 多项式
    
    ax2.semilogy(L_values, chi_mps, 'o-', markersize=8, linewidth=2, label='MPS')
    ax2.semilogy(L_values, chi_mera, 's-', markersize=8, linewidth=2, label='MERA')
    ax2.set_xlabel('System Size L', fontsize=12)
    ax2.set_ylabel('Required Bond Dimension χ', fontsize=12)
    ax2.set_title('Computational Cost Scaling',
                 fontsize=13, fontweight='bold')
    ax2.set_xscale('log')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.show()
    
    print("\n纠缠熵标度比较")
    print("="*60)
    print(f"{'L':<10} {'S(MPS)':<15} {'S(MERA)':<15} {'χ(MPS)':<15} {'χ(MERA)':<15}")
    print("-"*60)
    for i, L in enumerate(L_values[::2]):
        print(f"{L:<10} {S_mps[i*2]:<15.2f} {S_mera[i*2]:<15.2f} "
              f"{chi_mps[i*2]:<15.1f} {chi_mera[i*2]:<15.1f}")
    
    print("\nMERA优势：")
    print("- 纠缠熵: O(log log L) vs O(log L)")
    print("- 键维度: poly(log L) vs exp(log L)")
    print("- 计算成本低得多！")

compare_entanglement_scaling()

## 总结

### MERA关键概念

1. **层次结构**
   - 等距张量：粗粒化
   - 解纠缠器：去除短程纠缠
   - 多层：对应不同能量尺度

2. **实空间RG**
   - 显式实现重整化群
   - 临界点→RG不动点
   - 每层相似结构

3. **纠缠效率**
   - 临界系统：S ~ log log L
   - 键维度：poly(log L)
   - 远优于MPS

### 下一步

- 变分优化MERA
- 提取CFT数据
- 计算关联函数
- 识别临界点

## 练习

1. 验证不同χ的等距性
2. 实现3-to-1分支MERA
3. 计算粗粒化后的能量
4. 探索不同初始化方法

## 参考文献

1. Vidal, G. (2007). *Entanglement Renormalization*. PRL.
2. Vidal, G. (2008). *Class of Quantum Many-Body States*. PRL.
3. Evenbly, G. & Vidal, G. (2009). *Algorithms for Entanglement Renormalization*. PRB.